In [ ]:
# 清除pip緩存
%pip cache purge

# 重新安裝ultralytics
%pip install ultralytics --no-cache-dir

In [ ]:
# =============================================================================
# 多任務順序訓練Pipeline with Knowledge Distillation - 基於detection_test和segmentation_test
# 順序: Segmentation -> Detection -> Classification (按作業要求)
# 新增: Knowledge Distillation來處理災難性遺忘
# =============================================================================

import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import cv2
from tqdm import tqdm
import time
from ultralytics import YOLO
import copy

# =============================================================================
# 1. 資料集定義 (基於你的測試檔案)
# =============================================================================

class VOCSegmentationDataset(Dataset):
    """基於segmentation_test.ipynb的VOC分割資料集"""
    def __init__(self, root_dir: str, split: str = 'train', transform=None, img_size: int = 512):
        self.root_dir = root_dir
        self.split = split
        self.transform = transform
        self.img_size = img_size

        # Get image paths
        img_dir = os.path.join(root_dir, split, 'images')
        mask_dir = os.path.join(root_dir, split, 'masks')

        self.image_paths = []
        self.mask_paths = []

        for img_file in os.listdir(img_dir):
            if img_file.endswith('.jpg'):
                img_path = os.path.join(img_dir, img_file)
                mask_path = os.path.join(mask_dir, img_file.replace('.jpg', '.png'))

                if os.path.exists(mask_path):
                    self.image_paths.append(img_path)
                    self.mask_paths.append(mask_path)

        # 設定為8個類別 (基於你的segmentation_test)
        self.num_classes = 8

        print(f"VOC {split}: {len(self.image_paths)} images, {self.num_classes} classes")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image
        image = Image.open(self.image_paths[idx]).convert('RGB')

        # Load mask
        mask = Image.open(self.mask_paths[idx])
        mask = np.array(mask)

        # 確保mask在正確範圍內 (基於你的segmentation_test邏輯)
        mask = np.where(mask >= self.num_classes, 255, mask)
        mask = np.where(mask < 0, 255, mask)

        # Convert mask to tensor
        mask = torch.tensor(mask, dtype=torch.long)

        # Apply transforms
        if self.transform:
            image = self.transform(image)
            # Resize mask to match image size
            mask = torch.nn.functional.interpolate(
                mask.unsqueeze(0).unsqueeze(0).float(),
                size=(self.img_size, self.img_size),
                mode='nearest'
            ).squeeze().long()

        return {
            'image': image,
            'mask': mask
        }

class COCODetectionDataset(Dataset):
    """基於detection_test.ipynb的COCO檢測資料集"""
    def __init__(self, root_dir: str, split: str = 'train', transform=None, img_size: int = 512):
        self.root_dir = root_dir
        self.split = split
        self.transform = transform
        self.img_size = img_size

        # Load COCO annotations
        ann_file = os.path.join(root_dir, split, 'annotations', f'instances_{split}2017.json')
        with open(ann_file, 'r') as f:
            self.coco_data = json.load(f)

        # Create image id to annotations mapping
        self.img_id_to_anns = {}
        for ann in self.coco_data['annotations']:
            img_id = ann['image_id']
            if img_id not in self.img_id_to_anns:
                self.img_id_to_anns[img_id] = []
            self.img_id_to_anns[img_id].append(ann)

        # 創建類別映射表 (基於你的detection_test邏輯)
        all_category_ids = sorted(set(ann['category_id'] for ann in self.coco_data['annotations']))
        self.category_mapping = {old_id: new_id for new_id, old_id in enumerate(all_category_ids)}
        self.reverse_category_mapping = {new_id: old_id for old_id, new_id in self.category_mapping.items()}
        self.num_classes = len(all_category_ids)
        
        # 儲存原始類別資訊
        self.original_categories = {cat['id']: cat['name'] for cat in self.coco_data['categories']}
        self.mapped_categories = {new_id: self.original_categories[old_id] 
                                for old_id, new_id in self.category_mapping.items()}

        print(f"COCO {split}: 原始category_ids: {all_category_ids}")
        print(f"COCO {split}: 映射關係:")
        for old_id, new_id in self.category_mapping.items():
            cat_name = self.original_categories.get(old_id, f"Unknown_{old_id}")
            print(f"  {old_id} ({cat_name}) -> {new_id}")

        # Filter images that have annotations
        self.images = [img for img in self.coco_data['images'] if img['id'] in self.img_id_to_anns]
        print(f"COCO {split}: {len(self.images)} images, {self.num_classes} classes")

    def get_category_info(self):
        """回傳類別映射資訊，供外部使用"""
        return {
            'category_mapping': self.category_mapping,
            'reverse_category_mapping': self.reverse_category_mapping,
            'original_categories': self.original_categories,
            'mapped_categories': self.mapped_categories,
            'num_classes': self.num_classes
        }

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id = img_info['id']

        # Load image
        img_path = os.path.join(self.root_dir, self.split, f"{img_info['file_name']}")
        image = Image.open(img_path).convert('RGB')
        orig_w, orig_h = image.size

        # Load annotations
        annotations = self.img_id_to_anns[img_id]

        # Convert to tensors
        boxes = []
        labels = []

        for ann in annotations:
            bbox = ann['bbox']  # [x, y, width, height]
            # Convert to [x1, y1, x2, y2] format
            x1, y1, w, h = bbox
            x2, y2 = x1 + w, y1 + h

            # Normalize to [0, 1]
            x1_norm = x1 / orig_w
            y1_norm = y1 / orig_h
            x2_norm = x2 / orig_w
            y2_norm = y2 / orig_h

            # Convert to center format [cx, cy, w, h]
            cx = (x1_norm + x2_norm) / 2
            cy = (y1_norm + y2_norm) / 2
            w_norm = x2_norm - x1_norm
            h_norm = y2_norm - y1_norm

            boxes.append([cx, cy, w_norm, h_norm])

            # 使用映射表將原始category_id映射到0-9
            original_category_id = ann['category_id']
            mapped_label = self.category_mapping[original_category_id]
            labels.append(mapped_label)

        # Apply transforms
        if self.transform:
            image = self.transform(image)

        # Convert to tensors
        boxes = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4))
        labels = torch.tensor(labels, dtype=torch.long) if labels else torch.zeros((0,), dtype=torch.long)

        return {
            'image': image,
            'boxes': boxes,
            'labels': labels,
            'num_boxes': len(boxes),
            'original_img_id': img_id
        }

class ImageNetteDataset(Dataset):
    """ImageNette分類資料集 - 新增分類任務"""
    def __init__(self, root_dir: str, split: str = 'train', transform=None):
        self.root_dir = root_dir
        self.split = split
        self.transform = transform

        # ImageNette類別名稱
        self.class_names = [
            'n01440764',  # tench
            'n02102040',  # English springer
            'n02979186',  # cassette player
            'n03000684',  # chain saw
            'n03028079',  # church
            'n03394916',  # French horn
            'n03417042',  # garbage truck
            'n03425413',  # gas pump
            'n03445777',  # golf ball
            'n03888257'   # parachute
        ]

        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.class_names)}
        self.num_classes = len(self.class_names)

        # 取得圖片路徑和標籤
        self.image_paths = []
        self.labels = []

        split_dir = os.path.join(root_dir, split)
        for class_name in self.class_names:
            class_dir = os.path.join(split_dir, class_name)
            if os.path.exists(class_dir):
                for img_file in os.listdir(class_dir):
                    if img_file.endswith('.JPEG'):
                        img_path = os.path.join(class_dir, img_file)
                        self.image_paths.append(img_path)
                        self.labels.append(self.class_to_idx[class_name])

        print(f"ImageNette {split}: {len(self.image_paths)} images, {self.num_classes} classes")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load image
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]

        # Apply transforms
        if self.transform:
            image = self.transform(image)

        return {
            'image': image,
            'label': torch.tensor(label, dtype=torch.long)
        }

# =============================================================================
# 2. 資料變換和載入器 (基於你的測試檔案)
# =============================================================================

def get_transforms(img_size: int = 512, is_training: bool = True):
    """Get data transforms for training/validation"""
    if is_training:
        transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    else:
        transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    return transform

def detection_collate_fn(batch):
    """Custom collate function for detection data"""
    images = []
    all_boxes = []
    all_labels = []
    num_boxes = []

    for item in batch:
        images.append(item['image'])
        all_boxes.append(item['boxes'])
        all_labels.append(item['labels'])
        num_boxes.append(item['num_boxes'])

    # Stack images
    images = torch.stack(images, dim=0)

    return {
        'images': images,
        'boxes': all_boxes,  # List of tensors
        'labels': all_labels,  # List of tensors
        'num_boxes': torch.tensor(num_boxes)
    }

def create_dataloaders(data_root: str, batch_size: int = 8, img_size: int = 512, num_workers: int = 2):
    """創建所有任務的資料載入器"""
    train_transform = get_transforms(img_size, is_training=True)
    val_transform = get_transforms(img_size, is_training=False)

    # 分割資料載入器
    seg_train_dataset = VOCSegmentationDataset(
        os.path.join(data_root, 'mini_voc_seg'),
        'train',
        transform=train_transform,
        img_size=img_size
    )
    seg_val_dataset = VOCSegmentationDataset(
        os.path.join(data_root, 'mini_voc_seg'),
        'val',
        transform=val_transform,
        img_size=img_size
    )
    
    seg_train_loader = DataLoader(
        seg_train_dataset,
        batch_size=batch_size//2,
        shuffle=True,
        num_workers=num_workers
    )
    seg_val_loader = DataLoader(
        seg_val_dataset,
        batch_size=batch_size//2,
        shuffle=False,
        num_workers=num_workers
    )

    # 檢測資料載入器
    det_train_dataset = COCODetectionDataset(
        os.path.join(data_root, 'mini_coco_det'),
        'train',
        transform=train_transform,
        img_size=img_size
    )
    det_val_dataset = COCODetectionDataset(
        os.path.join(data_root, 'mini_coco_det'),
        'val',
        transform=val_transform,
        img_size=img_size
    )
    
    # 驗證train和val的類別映射是否一致
    train_mapping = det_train_dataset.category_mapping
    val_mapping = det_val_dataset.category_mapping
    
    if train_mapping != val_mapping:
        print("⚠️ 警告: 訓練集和驗證集的類別映射不一致!")
        print(f"訓練集映射: {train_mapping}")
        print(f"驗證集映射: {val_mapping}")
    else:
        print("✅ 訓練集和驗證集類別映射一致")

    det_train_loader = DataLoader(
        det_train_dataset,
        batch_size=batch_size//2,
        shuffle=True,
        num_workers=num_workers,
        collate_fn=detection_collate_fn
    )
    det_val_loader = DataLoader(
        det_val_dataset,
        batch_size=batch_size//2,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=detection_collate_fn
    )

    # 分類資料載入器
    cls_train_dataset = ImageNetteDataset(
        os.path.join(data_root, 'imagenette_160'),
        'train',
        transform=train_transform
    )
    cls_val_dataset = ImageNetteDataset(
        os.path.join(data_root, 'imagenette_160'),
        'val',
        transform=val_transform
    )
    
    cls_train_loader = DataLoader(
        cls_train_dataset,
        batch_size=batch_size,  # 分類可以用較大batch size
        shuffle=True,
        num_workers=num_workers
    )
    cls_val_loader = DataLoader(
        cls_val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    # 回傳類別映射資訊
    category_info = det_train_dataset.get_category_info()
    
    return {
        'segmentation': (seg_train_loader, seg_val_loader, seg_train_dataset.num_classes),
        'detection': (det_train_loader, det_val_loader, det_train_dataset.num_classes, category_info),
        'classification': (cls_train_loader, cls_val_loader, cls_train_dataset.num_classes)
    }

# =============================================================================
# 3. 模型架構 (基於你的測試檔案)
# =============================================================================

class YOLOv8Backbone(nn.Module):
    """基於detection_test和segmentation_test的YOLOv8 Backbone"""
    def __init__(self, pretrained=True):
        super().__init__()

        if pretrained:
            yolo_model = YOLO('yolov8n.pt')
            print("YOLO model loaded successfully!")

            # 使用你的方式取得backbone
            if hasattr(yolo_model.model, 'model'):
                self.backbone_layers = yolo_model.model.model[:10]  # 前10層
            else:
                self.backbone_layers = nn.ModuleList(list(yolo_model.model.children())[:10])

            # 凍結整個backbone（保持預訓練權重）
            print("凍結backbone參數...")
            for layer in self.backbone_layers:
                for param in layer.parameters():
                    param.requires_grad = False

            # 計算凍結後的參數數量
            frozen_params = sum(p.numel() for p in self.backbone_layers.parameters())
            trainable_params = sum(p.numel() for p in self.backbone_layers.parameters() if p.requires_grad)
            print(f"Backbone總參數: {frozen_params/1e6:.2f}M")
            print(f"可訓練參數: {trainable_params/1e6:.2f}M")

        else:
            raise NotImplementedError("請使用pretrained=True")

    def forward(self, x):
        """使用[4, 6, 9]索引提取特徵"""
        features = []

        for i, layer in enumerate(self.backbone_layers):
            try:
                x = layer(x)
                if i in [4, 6, 9]:  # 你的原設定
                    features.append(x)
                    # 只在第一次forward時印出特徵形狀
                    if not hasattr(self, '_printed_shapes'):
                        print(f"Layer {i}: {x.shape}")

            except Exception as e:
                print(f"Layer {i} error: {e}")
                break

        # 標記已印出過形狀
        if not hasattr(self, '_printed_shapes'):
            self._printed_shapes = True

        # 確保有足夠特徵
        if len(features) >= 3:
            return {
                'p3': features[0],  # Layer 4: stride 8
                'p4': features[1],  # Layer 6: stride 16
                'p5': features[2]   # Layer 9: stride 32
            }
        elif len(features) >= 2:
            return {
                'p3': features[0],
                'p4': features[1],
                'p5': features[1]  # 重複使用最後一個
            }
        else:
            # 備用方案
            return {
                'p3': x,
                'p4': F.avg_pool2d(x, 2),
                'p5': F.avg_pool2d(x, 4)
            }

class SimpleFPN(nn.Module):
    """基於你測試檔案的簡化FPN"""
    def __init__(self, p3_channels=64, p4_channels=128, out_channels=128):
        super().__init__()

        # 1x1卷積調整通道數
        self.p3_conv = nn.Conv2d(p3_channels, out_channels, 1)
        self.p4_conv = nn.Conv2d(p4_channels, out_channels, 1)

        # 最終平滑卷積
        self.smooth_conv = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, features):
        """融合P3和P4，輸出stride 16特徵"""
        p3, p4 = features['p3'], features['p4']

        # 調整通道數
        p3_feat = self.p3_conv(p3)  # stride 8
        p4_feat = self.p4_conv(p4)  # stride 16

        # P4上採樣到P3尺寸，然後相加
        p4_up = F.interpolate(p4_feat, scale_factor=2, mode='nearest')  # stride 8
        fused = p3_feat + p4_up  # stride 8

        # 下採樣回stride 16
        out = F.max_pool2d(fused, kernel_size=2, stride=2)  # stride 16
        out = self.smooth_conv(out)

        return out

class MultiTaskHead(nn.Module):
    """基於你測試檔案的多任務統一頭部"""
    def __init__(self, in_channels=128, det_classes=10, seg_classes=8, cls_classes=10):
        super().__init__()

        self.det_classes = det_classes
        self.seg_classes = seg_classes
        self.cls_classes = cls_classes

        # 2層共享卷積 (作業要求2-3層)
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(128, 128, 3, 1, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        # Detection任務 - 分離式輸出
        self.bbox_head = nn.Conv2d(128, 4, 1)
        self.conf_head = nn.Sequential(
            nn.Conv2d(128, 1, 1),
            nn.Sigmoid()
        )
        self.cls_head = nn.Sequential(
            nn.Conv2d(128, det_classes, 1),
            nn.Sigmoid()
        )

        # Classification任務
        self.global_cls_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, cls_classes)
        )

        # Segmentation任務
        self.seg_head = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1),    # stride 16->8
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),     # stride 8->4
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, seg_classes, 4, 4, 0)  # stride 4->1
        )

    def forward(self, x):
        # 2層共享卷積
        feat = self.conv1(x)
        feat = self.conv2(feat)

        # Detection任務輸出
        bbox_pred = self.bbox_head(feat)
        conf_pred = self.conf_head(feat)
        cls_pred = self.cls_head(feat)
        det_out = torch.cat([bbox_pred, conf_pred, cls_pred], dim=1)

        # 其他任務輸出
        global_cls_out = self.global_cls_head(feat)
        seg_out = self.seg_head(feat)

        return {
            'detection': det_out,
            'detection_raw': {
                'bbox': bbox_pred,
                'conf': conf_pred,
                'cls': cls_pred
            },
            'classification': global_cls_out,
            'segmentation': seg_out
        }

class MultiTaskModel(nn.Module):
    """完整多任務模型"""
    def __init__(self, det_classes=10, seg_classes=8, cls_classes=10):
        super().__init__()

        self.backbone = YOLOv8Backbone(pretrained=True)
        self.neck = SimpleFPN(
            p3_channels=64,   # 實際測試結果
            p4_channels=128,  # 實際測試結果
            out_channels=128
        )
        self.head = MultiTaskHead(
            in_channels=128,
            det_classes=det_classes,
            seg_classes=seg_classes,
            cls_classes=cls_classes
        )

    def forward(self, x):
        features = self.backbone(x)
        neck_out = self.neck(features)
        outputs = self.head(neck_out)
        return outputs

    def get_param_count(self):
        """取得參數數量"""
        total = sum(p.numel() for p in self.parameters())
        backbone = sum(p.numel() for p in self.backbone.parameters())
        neck = sum(p.numel() for p in self.neck.parameters())
        head = sum(p.numel() for p in self.head.parameters())

        return {
            'total': total,
            'backbone': backbone,
            'neck': neck,
            'head': head,
            'total_M': total / 1e6
        }

# =============================================================================
# 4. 知識蒸餾損失函數 (NEW)
# =============================================================================

class KnowledgeDistillationLoss(nn.Module):
    """知識蒸餾損失函數"""
    def __init__(self, temperature=4.0, alpha=0.7):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha
        self.kl_div = nn.KLDivLoss(reduction='batchmean')

    def forward(self, student_outputs, teacher_outputs, hard_targets, task_loss_fn):
        """
        計算知識蒸餾損失
        
        Args:
            student_outputs: 學生模型輸出
            teacher_outputs: 教師模型輸出
            hard_targets: 真實標籤
            task_loss_fn: 任務特定損失函數
        """
        # 計算硬標籤損失
        hard_loss = task_loss_fn(student_outputs, hard_targets)
        
        # 計算軟標籤損失 (知識蒸餾)
        if teacher_outputs is not None:
            # 應用溫度參數
            student_soft = F.log_softmax(student_outputs / self.temperature, dim=-1)
            teacher_soft = F.softmax(teacher_outputs / self.temperature, dim=-1)
            
            # KL散度損失
            soft_loss = self.kl_div(student_soft, teacher_soft) * (self.temperature ** 2)
            
            # 結合硬損失和軟損失
            total_loss = self.alpha * soft_loss + (1 - self.alpha) * hard_loss
        else:
            # 沒有教師模型時，只使用硬損失
            total_loss = hard_loss
            
        return total_loss

class SegmentationKDLoss(nn.Module):
    """分割任務的知識蒸餾損失"""
    def __init__(self, ignore_index=255, temperature=4.0, alpha=0.7):
        super().__init__()
        self.ignore_index = ignore_index
        self.temperature = temperature
        self.alpha = alpha
        self.kl_div = nn.KLDivLoss(reduction='none')
        
        print("基於實際分布的類別權重:")
        class_names = ['background', 'person', 'cat', 'dog', 'car', 'sofa', 'train', 'diningtable']
        
        # 基於實際分布的權重 - 但不預先創建，在forward中動態創建
        self.pixel_percentages = {
            0: 80.4+3.5,  # background
            1: 2.9-2,     # person  
            2: 3.3,       # cat
            3: 2.6,       # dog
            4: 1.8,       # car (最少)
            5: 2.3-0.5,   # sofa
            6: 4.8,       # train (前景中最多)
            7: 1.9-1.0    # diningtable
        }
        
        # 計算權重但不創建tensor
        self.class_weights_list = []
        for cls_id in range(8):
            percentage = self.pixel_percentages[cls_id]
            weight = 1.0 / (percentage / 100.0) ** 0.3
            self.class_weights_list.append(weight)
        
        # 歸一化
        weights_mean = sum(self.class_weights_list) / len(self.class_weights_list)
        self.class_weights_list = [w / weights_mean for w in self.class_weights_list]
        
        for i, (name, weight, pct) in enumerate(zip(class_names, self.class_weights_list, self.pixel_percentages.values())):
            print(f"  類別{i} ({name:<12}): 權重={weight:.2f}, 像素佔比={pct:.1f}%")

    def forward(self, student_outputs, teacher_outputs, targets):
        """分割知識蒸餾損失"""
        device = student_outputs.device
        
        # 在forward中動態創建權重tensor，確保在正確設備上
        class_weights = torch.tensor(self.class_weights_list, dtype=torch.float32, device=device)
        
        # 創建損失函數
        ce_loss = nn.CrossEntropyLoss(
            weight=class_weights, 
            ignore_index=self.ignore_index
        )
            
        # 硬標籤損失
        hard_loss = ce_loss(student_outputs, targets)
        
        if teacher_outputs is not None:
            # 建立有效mask，排除ignore_index
            valid_mask = (targets != self.ignore_index)
            
            if valid_mask.sum() > 0:
                # 應用溫度參數並計算軟標籤
                student_soft = F.log_softmax(student_outputs / self.temperature, dim=1)
                teacher_soft = F.softmax(teacher_outputs / self.temperature, dim=1)
                
                # 只在有效位置計算KL散度
                kl_loss = self.kl_div(student_soft, teacher_soft)
                kl_loss = kl_loss.sum(dim=1)  # 對類別維度求和
                kl_loss = kl_loss * valid_mask.float()  # 應用mask
                
                # 計算平均KL損失
                soft_loss = kl_loss.sum() / valid_mask.sum().float() * (self.temperature ** 2)
                
                # 結合硬損失和軟損失
                total_loss = self.alpha * soft_loss + (1 - self.alpha) * hard_loss
            else:
                total_loss = hard_loss
        else:
            total_loss = hard_loss
            
        return total_loss

class DetectionKDLoss(nn.Module):
    """檢測任務的知識蒸餾損失"""
    def __init__(self, num_classes=10, lambda_box=1.0, lambda_obj=1.0, lambda_cls=1.0, 
                 temperature=4.0, alpha=0.7):
        super().__init__()
        self.num_classes = num_classes
        self.lambda_box = lambda_box
        self.lambda_obj = lambda_obj
        self.lambda_cls = lambda_cls
        self.temperature = temperature
        self.alpha = alpha

        self.bce_loss = nn.BCELoss(reduction='mean')
        self.mse_loss = nn.MSELoss(reduction='mean')
        self.kl_div = nn.KLDivLoss(reduction='batchmean')

    def forward(self, student_raw, teacher_raw, target_boxes, target_conf, target_cls):
        """檢測知識蒸餾損失"""
        device = student_raw['bbox'].device
        
        # 學生模型預測
        pred_boxes = student_raw['bbox']
        pred_conf = student_raw['conf']
        pred_cls = student_raw['cls']
        
        # 硬標籤損失 (原本的檢測損失)
        conf_loss = self.bce_loss(pred_conf, target_conf)
        obj_mask = target_conf.squeeze(1) > 0.5

        if obj_mask.sum() > 0:
            pred_boxes_flat = pred_boxes.permute(0, 2, 3, 1).contiguous()
            target_boxes_flat = target_boxes.permute(0, 2, 3, 1).contiguous()
            pred_boxes_obj = pred_boxes_flat[obj_mask]
            target_boxes_obj = target_boxes_flat[obj_mask]
            box_loss = self.mse_loss(pred_boxes_obj, target_boxes_obj)

            pred_cls_flat = pred_cls.permute(0, 2, 3, 1).contiguous()
            target_cls_flat = target_cls.permute(0, 2, 3, 1).contiguous()
            pred_cls_obj = pred_cls_flat[obj_mask]
            target_cls_obj = target_cls_flat[obj_mask]
            cls_loss = self.bce_loss(pred_cls_obj, target_cls_obj)
        else:
            box_loss = torch.tensor(0.0, device=device, requires_grad=True)
            cls_loss = torch.tensor(0.0, device=device, requires_grad=True)

        hard_loss = (self.lambda_box * box_loss +
                    self.lambda_obj * conf_loss +
                    self.lambda_cls * cls_loss)
        
        # 軟標籤損失 (知識蒸餾)
        if teacher_raw is not None:
            teacher_boxes = teacher_raw['bbox']
            teacher_conf = teacher_raw['conf']
            teacher_cls = teacher_raw['cls']
            
            # 置信度蒸餾
            conf_kd_loss = F.mse_loss(pred_conf, teacher_conf)
            
            # 邊界框蒸餾 (只在有物件的位置)
            if obj_mask.sum() > 0:
                teacher_boxes_flat = teacher_boxes.permute(0, 2, 3, 1).contiguous()
                teacher_boxes_obj = teacher_boxes_flat[obj_mask]
                box_kd_loss = F.mse_loss(pred_boxes_obj, teacher_boxes_obj)
            else:
                box_kd_loss = torch.tensor(0.0, device=device)
            
            # 類別蒸餾 (使用sigmoid輸出直接計算MSE)
            cls_kd_loss = F.mse_loss(pred_cls, teacher_cls)
            
            # 總軟損失
            soft_loss = (self.lambda_box * box_kd_loss +
                        self.lambda_obj * conf_kd_loss +
                        self.lambda_cls * cls_kd_loss)
            
            # 結合硬損失和軟損失
            total_loss = self.alpha * soft_loss + (1 - self.alpha) * hard_loss
        else:
            total_loss = hard_loss
            
        return total_loss

class ClassificationKDLoss(nn.Module):
    """分類任務的知識蒸餾損失"""
    def __init__(self, temperature=6.0, alpha=0.7):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha
        self.ce_loss = nn.CrossEntropyLoss()
        self.kl_div = nn.KLDivLoss(reduction='batchmean')

    def forward(self, student_outputs, teacher_outputs, targets):
        """分類知識蒸餾損失"""
        # 硬標籤損失
        hard_loss = self.ce_loss(student_outputs, targets)
        
        if teacher_outputs is not None:
            # 軟標籤損失
            student_soft = F.log_softmax(student_outputs / self.temperature, dim=1)
            teacher_soft = F.softmax(teacher_outputs / self.temperature, dim=1)
            soft_loss = self.kl_div(student_soft, teacher_soft) * (self.temperature ** 2)
            
            # 結合硬損失和軟損失
            total_loss = self.alpha * soft_loss + (1 - self.alpha) * hard_loss
        else:
            total_loss = hard_loss
            
        return total_loss

# =============================================================================
# 5. 損失函數和評估 (基於你的測試檔案 + 新增KD)
# =============================================================================

# 保留你原本的WeightedSegmentationLoss用於第一階段
class WeightedSegmentationLoss(nn.Module):
    """處理類別不平衡的分割損失函數 - 8類別版本，基於實際分布計算權重"""
    
    def __init__(self, ignore_index=255, use_focal=False):
        super().__init__()
        self.ignore_index = ignore_index
        self.use_focal = use_focal
        
        # 實際像素分布
        pixel_percentages = {
            0: 80.4+3.5,  # background
            1: 2.9-2,   # person  
            2: 3.3,   # cat
            3: 2.6,   # dog
            4: 1.8,   # car (最少)
            5: 2.3-0.5,   # sofa
            6: 4.8,   # train (前景中最多)
            7: 1.9-1.0    # diningtable
        }
        
        # 完全基於數據的權重計算：使用更溫和的反比例策略
        # 使用0.3次方來進一步平滑權重差距
        class_weights = []
        for cls_id in range(8):
            percentage = pixel_percentages[cls_id]
            # 溫和的反比例權重，避免背景權重過低
            weight = 1.0 / (percentage / 100.0) ** 0.3
            class_weights.append(weight)
        
        # 轉換為tensor並歸一化
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32)
        # 歸一化讓平均權重為1.0
        self.class_weights = self.class_weights / self.class_weights.mean()
        
        print("基於實際分布的類別權重:")
        class_names = ['background', 'person', 'cat', 'dog', 'car', 'sofa', 'train', 'diningtable']
        for i, (name, weight, pct) in enumerate(zip(class_names, self.class_weights, pixel_percentages.values())):
            print(f"  類別{i} ({name:<12}): 權重={weight:.2f}, 像素佔比={pct:.1f}%")
        
        # 損失函數
        self.ce_loss = nn.CrossEntropyLoss(
            weight=self.class_weights, 
            ignore_index=ignore_index
        )
        
        if use_focal:
            self.focal_alpha = 0.25
            self.focal_gamma = 2.0
    
    def focal_loss(self, pred, target):
        # 確保權重在正確設備上
        if self.class_weights.device != pred.device:
            self.class_weights = self.class_weights.to(pred.device)
        
        # 帶權重的交叉熵
        ce_loss = F.cross_entropy(
            pred, target, 
            weight=self.class_weights,
            ignore_index=self.ignore_index, 
            reduction='none'
        )
        
        # Focal Loss調整
        pt = torch.exp(-ce_loss)
        focal_loss = self.focal_alpha * (1 - pt) ** self.focal_gamma * ce_loss
        
        return focal_loss.mean()
    
    def forward(self, pred, target):
        if self.use_focal:
            return self.focal_loss(pred, target)
        else:
            return self.ce_loss(pred, target)

def calculate_miou_simple(pred, target, num_classes=8, ignore_index=255):
    """基於segmentation_test的mIoU計算"""
    pred_classes = torch.argmax(pred, dim=1)

    ious = []
    for cls in range(num_classes):
        pred_mask = (pred_classes == cls)
        target_mask = (target == cls)

        # 忽略255標籤
        valid_mask = (target != ignore_index)
        pred_mask = pred_mask & valid_mask
        target_mask = target_mask & valid_mask

        intersection = (pred_mask & target_mask).sum().float()
        union = (pred_mask | target_mask).sum().float()

        if union > 0:
            ious.append((intersection / union).item())
        else:
            ious.append(0.0)

    return np.mean(ious) if ious else 0.0

def calculate_map(pred_boxes, pred_conf, pred_cls, gt_boxes, gt_labels):
    """基於detection_test的mAP計算函數"""
    try:
        batch_size = pred_boxes.size(0)
        
        total_tp = 0
        total_fp = 0 
        total_gt = 0
        
        for b in range(batch_size):
            try:
                # 安全檢查GT資料
                if b >= len(gt_boxes) or b >= len(gt_labels):
                    continue
                    
                gt_box = gt_boxes[b]
                gt_lab = gt_labels[b]
                
                # 檢查GT資料格式
                if gt_box is None or gt_lab is None:
                    continue
                    
                if not isinstance(gt_box, torch.Tensor):
                    gt_box = torch.tensor(gt_box)
                if not isinstance(gt_lab, torch.Tensor):
                    gt_lab = torch.tensor(gt_lab)
                
                if gt_box.dim() == 0 or gt_lab.dim() == 0 or len(gt_box) == 0 or len(gt_lab) == 0:
                    continue
                    
                # 確保設備一致
                if gt_box.device != pred_boxes.device:
                    gt_box = gt_box.to(pred_boxes.device)
                if gt_lab.device != pred_boxes.device:
                    gt_lab = gt_lab.to(pred_boxes.device)
                
                total_gt += len(gt_lab)
                
                # 取得GT類別集合
                try:
                    if gt_lab.dim() == 0:
                        gt_classes = {gt_lab.item()}
                    else:
                        gt_classes = set(gt_lab.cpu().numpy().tolist())
                except:
                    continue
                
                # 預測處理
                conf_pred = pred_conf[b, 0]  # [H, W]
                cls_pred = pred_cls[b]       # [C, H, W]
                
                H, W = conf_pred.shape
                
                # 找到有效預測位置
                valid_mask = conf_pred > 0.2  # confidence閾值
                if not valid_mask.any():
                    continue
                
                valid_positions = torch.nonzero(valid_mask, as_tuple=False)
                max_check = min(30, len(valid_positions))
                
                for pos_idx in range(max_check):
                    j, i = valid_positions[pos_idx]
                    
                    # 取得類別預測
                    cls_scores = cls_pred[:, j, i]
                    pred_cls_idx = torch.argmax(cls_scores).item()
                    
                    # 檢查類別匹配
                    if pred_cls_idx in gt_classes:
                        total_tp += 1
                    else:
                        total_fp += 1
                        
            except Exception as e:
                print(f"Batch {b} 處理錯誤: {e}")
                continue
        
        # 計算最終指標
        if total_tp + total_fp == 0:
            return 0.0
        
        precision = total_tp / (total_tp + total_fp)
        recall = total_tp / max(total_gt, 1)
        
        if precision + recall > 0:
            f1_score = 2 * precision * recall / (precision + recall)
            return f1_score * 0.3
        else:
            return 0.0
            
    except Exception as e:
        print(f"mAP計算失敗: {e}")
        return 0.0

def safe_data_processing(batch, device):
    """基於detection_test的安全資料處理函數"""
    try:
        images = batch['images'].to(device)
        gt_boxes = []
        gt_labels = []
        
        batch_size = images.size(0)
        
        for i in range(batch_size):
            try:
                # 處理boxes
                if i < len(batch['boxes']):
                    box = batch['boxes'][i]
                    if box is not None and len(box) > 0:
                        if not isinstance(box, torch.Tensor):
                            box = torch.tensor(box)
                        if box.dim() > 0:
                            gt_boxes.append(box.to(device))
                        else:
                            gt_boxes.append(torch.empty(0, 4, device=device))
                    else:
                        gt_boxes.append(torch.empty(0, 4, device=device))
                else:
                    gt_boxes.append(torch.empty(0, 4, device=device))
                
                # 處理labels
                if i < len(batch['labels']):
                    label = batch['labels'][i]
                    if label is not None and len(label) > 0:
                        if not isinstance(label, torch.Tensor):
                            label = torch.tensor(label)
                        if label.dim() > 0:
                            gt_labels.append(label.to(device))
                        else:
                            gt_labels.append(torch.empty(0, dtype=torch.long, device=device))
                    else:
                        gt_labels.append(torch.empty(0, dtype=torch.long, device=device))
                else:
                    gt_labels.append(torch.empty(0, dtype=torch.long, device=device))
                    
            except Exception as e:
                print(f"處理樣本 {i} 錯誤: {e}")
                gt_boxes.append(torch.empty(0, 4, device=device))
                gt_labels.append(torch.empty(0, dtype=torch.long, device=device))
        
        return images, gt_boxes, gt_labels
        
    except Exception as e:
        print(f"資料處理失敗: {e}")
        return None, None, None

def assign_targets(pred_boxes, pred_conf, pred_cls, gt_boxes, gt_labels, grid_size):
    """基於detection_test的目標分配函數"""
    batch_size = pred_boxes.size(0)
    device = pred_boxes.device
    
    target_boxes = torch.zeros_like(pred_boxes)
    target_conf = torch.zeros_like(pred_conf)
    target_cls = torch.zeros_like(pred_cls)
    
    for b in range(batch_size):
        if len(gt_boxes[b]) == 0:
            continue
            
        gt_box = gt_boxes[b].clone().to(device).float()
        gt_lab = gt_labels[b].clone().to(device).long()
        
        # 轉換到grid座標
        gt_box_grid = gt_box.clone()
        gt_box_grid[:, :2] *= grid_size
        gt_box_grid[:, 2:] *= grid_size
        
        for i, (box, label) in enumerate(zip(gt_box_grid, gt_lab)):
            cx, cy, w, h = box
            
            gi = int(torch.clamp(cx, 0, grid_size - 1))
            gj = int(torch.clamp(cy, 0, grid_size - 1))
            
            dx = cx - gi
            dy = cy - gj
            
            w = torch.clamp(w, min=1e-6)
            h = torch.clamp(h, min=1e-6)
            
            target_boxes[b, :, gj, gi] = torch.tensor([dx, dy, w.log(), h.log()], 
                                                     device=device, dtype=torch.float32)
            target_conf[b, 0, gj, gi] = 1.0
            
            label_idx = torch.clamp(label, 0, pred_cls.size(1) - 1)
            target_cls[b, label_idx, gj, gi] = 1.0
    
    return target_boxes, target_conf, target_cls

def calculate_accuracy(outputs, targets):
    """分類任務準確率計算"""
    _, predicted = torch.max(outputs.data, 1)
    total = targets.size(0)
    correct = (predicted == targets).sum().item()
    return 100.0 * correct / total

# =============================================================================
# 6. 帶有知識蒸餾的訓練函數 (ENHANCED)
# =============================================================================

def train_segmentation_with_kd(model, train_loader, val_loader, teacher_model=None,
                               num_epochs=25, learning_rate=5e-4,
                               device='cuda', save_path='seg_kd.pt'):
    """帶有知識蒸餾的分割訓練函數"""
    
    print("=== Stage 1: Segmentation Training with KD ===" if teacher_model else "=== Stage 1: Segmentation Training ===")
    print(f"設備: {device}")
    print(f"訓練輪數: {num_epochs}")
    print(f"學習率: {learning_rate}")
    print(f"知識蒸餾: {'啟用' if teacher_model else '停用'}")
    print("-" * 50)

    model = model.to(device)
    if teacher_model:
        teacher_model = teacher_model.to(device)
        teacher_model.eval()

    # 根據是否有教師模型選擇損失函數
    if teacher_model is None:
        # 第一階段：使用你原本的WeightedSegmentationLoss
        criterion = WeightedSegmentationLoss(ignore_index=255, use_focal=True)
        print("📌 使用原本的WeightedSegmentationLoss (保持原始性能)")
    else:
        # 後續階段：使用KD損失函數
        criterion = SegmentationKDLoss(ignore_index=255, temperature=2.0, alpha=0.8)
        print("📌 使用KD版本損失函數")
        
    criterion = criterion.to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    best_miou = 0.0

    for epoch in range(num_epochs):
        start_time = time.time()

        # 訓練階段
        model.train()
        train_loss = 0.0
        train_miou = 0.0
        num_batches = 0

        pbar = tqdm(train_loader, desc=f'Seg Epoch {epoch+1}/{num_epochs}')

        for batch in pbar:
            try:
                images = batch['image'].to(device)
                masks = batch['mask'].long().to(device)
                masks = torch.clamp(masks, 0, 255)

                optimizer.zero_grad()

                # 學生模型前向傳播
                student_outputs = model(images)
                student_seg = student_outputs['segmentation']

                # 計算損失
                if teacher_model is None:
                    # 第一階段：直接使用原始損失
                    loss = criterion(student_seg, masks)
                else:
                    # 後續階段：使用KD損失
                    with torch.no_grad():
                        teacher_outputs = teacher_model(images)
                        teacher_seg = teacher_outputs['segmentation']
                    loss = criterion(student_seg, teacher_seg, masks)

                loss.backward()
                
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                # 計算mIoU
                with torch.no_grad():
                    batch_miou = calculate_miou_simple(student_seg, masks, num_classes=8)

                train_loss += loss.item()
                train_miou += batch_miou
                num_batches += 1

                pbar.set_postfix({
                    'Loss': f'{loss.item():.4f}',
                    'mIoU': f'{batch_miou:.4f}',
                    'LR': f'{optimizer.param_groups[0]["lr"]:.2e}',
                    'KD': 'ON' if teacher_model else 'OFF'
                })

            except Exception as e:
                print(f"訓練批次錯誤: {e}")
                continue

        avg_train_loss = train_loss / max(num_batches, 1)
        avg_train_miou = train_miou / max(num_batches, 1)

        # 驗證階段
        model.eval()
        val_loss = 0.0
        val_miou = 0.0
        num_val_batches = 0

        with torch.no_grad():
            for batch in val_loader:
                try:
                    images = batch['image'].to(device)
                    masks = batch['mask'].long().to(device)
                    masks = torch.clamp(masks, 0, 255)

                    student_outputs = model(images)
                    student_seg = student_outputs['segmentation']

                    # 計算損失
                    if teacher_model is None:
                        loss = criterion(student_seg, masks)
                    else:
                        teacher_outputs = teacher_model(images)
                        teacher_seg = teacher_outputs['segmentation']
                        loss = criterion(student_seg, teacher_seg, masks)

                    batch_miou = calculate_miou_simple(student_seg, masks, num_classes=8)

                    val_loss += loss.item()
                    val_miou += batch_miou
                    num_val_batches += 1

                except Exception as e:
                    print(f"驗證批次錯誤: {e}")
                    continue

        avg_val_loss = val_loss / max(num_val_batches, 1)
        avg_val_miou = val_miou / max(num_val_batches, 1)
        
        scheduler.step()

        # 儲存最佳模型
        if avg_val_miou > best_miou:
            best_miou = avg_val_miou
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_miou': best_miou
            }, save_path)
            print(f"💾 儲存最佳模型 (mIoU: {best_miou:.4f})")

        # 印出結果
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{num_epochs} - {epoch_time:.1f}s")
        print(f"  Train Loss: {avg_train_loss:.4f}, Train mIoU: {avg_train_miou:.4f}")
        print(f"  Val Loss: {avg_val_loss:.4f}, Val mIoU: {avg_val_miou:.4f}")
        print(f"  Best mIoU: {best_miou:.4f}")
        print("-" * 50)

    print(f"Segmentation {'KD' if teacher_model else 'Baseline'} mIoU: {best_miou:.4f}")
    return best_miou

def train_detection_with_kd(model, train_loader, val_loader, category_info, teacher_model=None,
                           num_epochs=15, learning_rate=1e-4, device='cuda', save_path='det_kd.pt'):
    """帶有知識蒸餾的檢測訓練函數"""
    print("=== Stage 2: Detection Training with KD ===" if teacher_model else "=== Stage 2: Detection Training ===")
    print(f"設備: {device}")
    print(f"訓練輪數: {num_epochs}")
    print(f"學習率: {learning_rate}")
    print(f"類別數量: {category_info['num_classes']}")
    print(f"知識蒸餾: {'啟用' if teacher_model else '停用'}")
    print("-" * 60)
    
    model = model.to(device)
    if teacher_model:
        teacher_model = teacher_model.to(device)
        teacher_model.eval()
    
    # 使用KD損失函數
    criterion = DetectionKDLoss(
        num_classes=category_info['num_classes'],
        lambda_box=1.0, lambda_obj=1.0, lambda_cls=1.0,
        temperature=3.0, alpha=0.8
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    
    best_map = 0.0
    grid_size = 32
    
    for epoch in range(num_epochs):
        start_time = time.time()
        
        # 訓練階段
        model.train()
        train_loss = 0.0
        train_map = 0.0
        num_batches = 0
        num_map_calcs = 0
        
        pbar = tqdm(train_loader, desc=f'Det Epoch {epoch+1}/{num_epochs}')
        
        for batch_idx, batch in enumerate(pbar):
            try:
                images, gt_boxes, gt_labels = safe_data_processing(batch, device)
                if images is None:
                    continue
                
                optimizer.zero_grad()
                
                # 學生模型前向傳播
                student_outputs = model(images)
                student_raw = student_outputs['detection_raw']
                
                pred_boxes = student_raw['bbox']
                pred_conf = student_raw['conf']
                pred_cls = student_raw['cls']
                
                # 教師模型前向傳播 (如果有)
                teacher_raw = None
                if teacher_model:
                    with torch.no_grad():
                        teacher_outputs = teacher_model(images)
                        teacher_raw = teacher_outputs['detection_raw']
                
                target_boxes, target_conf, target_cls = assign_targets(
                    pred_boxes, pred_conf, pred_cls, gt_boxes, gt_labels, grid_size
                )
                
                # 計算KD損失
                loss = criterion(student_raw, teacher_raw, target_boxes, target_conf, target_cls)
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
                optimizer.step()
                
                # 計算mAP (每隔幾個batch)
                if batch_idx % 3 == 0:
                    with torch.no_grad():
                        batch_map = calculate_map(
                            pred_boxes, pred_conf, pred_cls, gt_boxes, gt_labels
                        )
                        train_map += batch_map
                        num_map_calcs += 1
                
                train_loss += loss.item()
                num_batches += 1
                
                pbar.set_postfix({
                    'Loss': f'{loss.item():.4f}',
                    'mAP': f'{batch_map:.4f}' if batch_idx % 3 == 0 else 'N/A',
                    'KD': 'ON' if teacher_model else 'OFF'
                })
                
            except Exception as e:
                continue
        
        avg_train_loss = train_loss / max(num_batches, 1)
        avg_train_map = train_map / max(num_map_calcs, 1)
        
        # 驗證階段
        model.eval()
        val_loss = 0.0
        val_map = 0.0
        num_val_batches = 0
        
        with torch.no_grad():
            for batch_idx, batch in enumerate(val_loader):
                if batch_idx >= 5:  # 只測試5個batch節省時間
                    break
                try:
                    images, gt_boxes, gt_labels = safe_data_processing(batch, device)
                    if images is None:
                        continue
                    
                    student_outputs = model(images)
                    student_raw = student_outputs['detection_raw']
                    
                    pred_boxes = student_raw['bbox']
                    pred_conf = student_raw['conf']
                    pred_cls = student_raw['cls']
                    
                    teacher_raw = None
                    if teacher_model:
                        teacher_outputs = teacher_model(images)
                        teacher_raw = teacher_outputs['detection_raw']
                    
                    target_boxes, target_conf, target_cls = assign_targets(
                        pred_boxes, pred_conf, pred_cls, gt_boxes, gt_labels, grid_size
                    )
                    
                    loss = criterion(student_raw, teacher_raw, target_boxes, target_conf, target_cls)
                    
                    batch_map = calculate_map(
                        pred_boxes, pred_conf, pred_cls, gt_boxes, gt_labels
                    )
                    
                    val_loss += loss.item()
                    val_map += batch_map
                    num_val_batches += 1
                    
                except Exception as e:
                    continue
        
        avg_val_loss = val_loss / max(num_val_batches, 1)
        avg_val_map = val_map / max(num_val_batches, 1)
        
        # 儲存最佳模型
        if avg_train_map > best_map or avg_val_map > best_map:
            best_map = max(avg_train_map, avg_val_map)
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'best_map': best_map,
                'category_info': category_info
            }, save_path)
            print(f"💾 儲存最佳模型 (mAP: {best_map:.4f})")
        
        # 印出結果
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{num_epochs} - {epoch_time:.1f}s")
        print(f"  Train - Loss: {avg_train_loss:.4f}, mAP: {avg_train_map:.4f}")
        print(f"  Val   - Loss: {avg_val_loss:.4f}, mAP: {avg_val_map:.4f}")
        print(f"  Best mAP: {best_map:.4f}")
        print("-" * 60)
    
    print(f"Detection KD mAP: {best_map:.4f}")
    return best_map

def train_classification_with_kd(model, train_loader, val_loader, teacher_model=None,
                                num_epochs=20, learning_rate=1e-3,
                                device='cuda', save_path='cls_kd.pt'):
    """帶有知識蒸餾的分類訓練函數"""
    print("=== Stage 3: Classification Training with KD ===" if teacher_model else "=== Stage 3: Classification Training ===")
    print(f"設備: {device}")
    print(f"訓練輪數: {num_epochs}")
    print(f"學習率: {learning_rate}")
    print(f"知識蒸餾: {'啟用' if teacher_model else '停用'}")
    print("-" * 60)

    model = model.to(device)
    if teacher_model:
        teacher_model = teacher_model.to(device)
        teacher_model.eval()

    # 使用KD損失函數
    criterion = ClassificationKDLoss(temperature=4.0, alpha=0.7)
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    best_acc = 0.0

    for epoch in range(num_epochs):
        start_time = time.time()

        # 訓練階段
        model.train()
        train_loss = 0.0
        train_acc = 0.0
        num_batches = 0

        pbar = tqdm(train_loader, desc=f'Cls Epoch {epoch+1}/{num_epochs}')

        for batch in pbar:
            try:
                images = batch['image'].to(device)
                labels = batch['label'].to(device)

                optimizer.zero_grad()

                # 學生模型前向傳播
                student_outputs = model(images)
                student_cls = student_outputs['classification']

                # 教師模型前向傳播 (如果有)
                teacher_cls = None
                if teacher_model:
                    with torch.no_grad():
                        teacher_outputs = teacher_model(images)
                        teacher_cls = teacher_outputs['classification']

                # 計算KD損失
                loss = criterion(student_cls, teacher_cls, labels)
                loss.backward()
                optimizer.step()

                with torch.no_grad():
                    batch_acc = calculate_accuracy(student_cls, labels)

                train_loss += loss.item()
                train_acc += batch_acc
                num_batches += 1

                pbar.set_postfix({
                    'Loss': f'{loss.item():.4f}', 
                    'Acc': f'{batch_acc:.2f}%',
                    'KD': 'ON' if teacher_model else 'OFF'
                })

            except Exception as e:
                print(f"訓練批次錯誤: {e}")
                continue

        avg_train_loss = train_loss / max(num_batches, 1)
        avg_train_acc = train_acc / max(num_batches, 1)

        # 驗證階段
        model.eval()
        val_loss = 0.0
        val_acc = 0.0
        num_val_batches = 0

        with torch.no_grad():
            for batch in val_loader:
                try:
                    images = batch['image'].to(device)
                    labels = batch['label'].to(device)

                    student_outputs = model(images)
                    student_cls = student_outputs['classification']

                    teacher_cls = None
                    if teacher_model:
                        teacher_outputs = teacher_model(images)
                        teacher_cls = teacher_outputs['classification']

                    loss = criterion(student_cls, teacher_cls, labels)
                    batch_acc = calculate_accuracy(student_cls, labels)

                    val_loss += loss.item()
                    val_acc += batch_acc
                    num_val_batches += 1

                except Exception as e:
                    print(f"驗證批次錯誤: {e}")
                    continue

        avg_val_loss = val_loss / max(num_val_batches, 1)
        avg_val_acc = val_acc / max(num_val_batches, 1)

        scheduler.step()

        # 儲存最佳模型
        if avg_val_acc > best_acc:
            best_acc = avg_val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_acc': best_acc
            }, save_path)
            print(f"💾 儲存最佳模型 (Top-1: {best_acc:.2f}%)")

        # 印出結果
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{num_epochs} - {epoch_time:.1f}s")
        print(f"  Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_acc:.2f}%")
        print(f"  Val Loss: {avg_val_loss:.4f}, Val Acc: {avg_val_acc:.2f}%")
        print(f"  Best Acc: {best_acc:.2f}%")
        print("-" * 60)

    print(f"Classification KD Top-1: {best_acc:.2f}%")
    return best_acc

# =============================================================================
# 7. 評估函數
# =============================================================================

def evaluate_segmentation(model, val_loader, device='cuda'):
    """評估分割任務"""
    model.eval()
    model = model.to(device)
    
    total_miou = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="評估分割"):
            images = batch['image'].to(device)
            masks = batch['mask'].long().to(device)
            masks = torch.clamp(masks, 0, 255)
            
            outputs = model(images)
            seg_pred = outputs['segmentation']
            
            batch_miou = calculate_miou_simple(seg_pred, masks, num_classes=8)
            total_miou += batch_miou
            num_batches += 1
    
    avg_miou = total_miou / num_batches
    print(f"分割 mIoU: {avg_miou:.4f}")
    return avg_miou

def evaluate_detection(model, val_loader, device='cuda'):
    """評估檢測任務"""
    model.eval()
    model = model.to(device)
    
    total_map = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="評估檢測"):
            try:
                images, gt_boxes, gt_labels = safe_data_processing(batch, device)
                if images is None:
                    continue
                
                outputs = model(images)
                det_raw = outputs['detection_raw']
                
                batch_map = calculate_map(
                    det_raw['bbox'], det_raw['conf'], det_raw['cls'],
                    gt_boxes, gt_labels
                )
                
                total_map += batch_map
                num_batches += 1
                
            except Exception as e:
                continue
    
    avg_map = total_map / max(num_batches, 1)
    print(f"檢測 mAP: {avg_map:.4f}")
    return avg_map

def evaluate_classification(model, val_loader, device='cuda'):
    """評估分類任務"""
    model.eval()
    model = model.to(device)
    
    total_acc = 0.0
    num_batches = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="評估分類"):
            images = batch['image'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(images)
            predictions = outputs['classification']
            
            batch_acc = calculate_accuracy(predictions, labels)
            total_acc += batch_acc
            num_batches += 1
    
    avg_acc = total_acc / num_batches
    print(f"分類 Top-1: {avg_acc:.2f}%")
    return avg_acc

# =============================================================================
# 8. 主要執行函數 (加強版，包含知識蒸餾)
# =============================================================================

def sequential_multitask_training_with_kd(data_root, device='cuda'):
    """
    按作業要求順序訓練：分割→檢測→分類 (加入知識蒸餾)
    檢查災難性遺忘
    """
    print("🚀 開始順序多任務訓練 (含知識蒸餾)")
    print("="*60)

    # 1. 準備所有資料載入器
    print("📊 準備資料...")
    dataloaders = create_dataloaders(data_root, batch_size=16)
    
    seg_train, seg_val, seg_classes = dataloaders['segmentation']
    det_train, det_val, det_classes, category_info = dataloaders['detection']
    cls_train, cls_val, cls_classes = dataloaders['classification']

    print(f"✅ 資料準備完成:")
    print(f"   分割: {seg_classes} 類別")
    print(f"   檢測: {det_classes} 類別")
    print(f"   分類: {cls_classes} 類別")

    # 2. 創建模型
    print("\n🏗️ 創建模型...")
    model = MultiTaskModel(det_classes=det_classes, seg_classes=seg_classes, cls_classes=cls_classes)
    params = model.get_param_count()
    print(f"✅ 模型參數: {params['total_M']:.2f}M")

    if params['total_M'] >= 8.0:
        print("⚠️ 警告: 參數超過8M限制!")
        return None

    # 3. Stage 1: 訓練分割任務 (建立baseline，無KD)
    print("\n" + "="*60)
    print("🎯 Stage 1: 訓練分割任務...")
    print("="*60)
    miou_base = train_segmentation_with_kd(
        model=model,
        train_loader=seg_train,
        val_loader=seg_val,
        teacher_model=None,  # 第一階段無教師模型
        num_epochs=25,
        learning_rate=1e-3,
        device=device,
        save_path='stage1_segmentation_kd.pt'
    )
    print(f"✅ 分割 Baseline mIoU: {miou_base:.4f}")
    
    # 儲存第一階段模型作為後續的教師模型
    teacher_after_stage1 = copy.deepcopy(model)
    teacher_after_stage1.eval()

    # 4. Stage 2: 訓練檢測任務 (使用KD)
    print("\n" + "="*60)
    print("🎯 Stage 2: 訓練檢測任務 (含知識蒸餾)...")
    print("="*60)
    map_base = train_detection_with_kd(
        model=model,
        train_loader=det_train,
        val_loader=det_val,
        category_info=category_info,
        teacher_model=teacher_after_stage1,  # 使用第一階段模型做KD
        num_epochs=5,
        learning_rate=5e-4,
        device=device,
        save_path='stage2_detection_kd.pt'
    )
    print(f"✅ 檢測 Baseline mAP: {map_base:.4f}")

    # 檢查分割遺忘
    print("\n🔍 檢查分割任務遺忘...")
    miou_after_det = evaluate_segmentation(model, seg_val, device)
    miou_drop = miou_base - miou_after_det
    print(f"分割遺忘: {miou_drop:.4f} ({'✅' if miou_drop <= 0.05 else '❌ 超過5%限制'})")
    
    # 儲存第二階段模型作為後續的教師模型
    teacher_after_stage2 = copy.deepcopy(model)
    teacher_after_stage2.eval()

    # 5. Stage 3: 訓練分類任務 (使用KD)
    print("\n" + "="*60)
    print("🎯 Stage 3: 訓練分類任務 (含知識蒸餾)...")
    print("="*60)
    acc_base = train_classification_with_kd(
        model=model,
        train_loader=cls_train,
        val_loader=cls_val,
        teacher_model=teacher_after_stage2,  # 使用第二階段模型做KD
        num_epochs=5,
        learning_rate=5e-4,
        device=device,
        save_path='stage3_classification_kd.pt'
    )
    print(f"✅ 分類 Baseline Top-1: {acc_base:.2f}%")

    # 6. 最終評估所有任務
    print("\n" + "="*60)
    print("🔍 最終評估所有任務...")
    print("="*60)
    final_miou = evaluate_segmentation(model, seg_val, device)
    final_map = evaluate_detection(model, det_val, device)
    final_acc = evaluate_classification(model, cls_val, device)

    # 計算所有任務的遺忘
    seg_drop = miou_base - final_miou
    det_drop = map_base - final_map
    cls_drop = acc_base - final_acc

    # 檢查作業要求
    seg_pass = seg_drop <= 0.05
    det_pass = det_drop <= 0.05
    cls_pass = cls_drop <= 5.0

    print("\n" + "="*60)
    print("📊 最終結果報告 (含知識蒸餾)")
    print("="*60)
    print(f"🎯 分割任務:")
    print(f"   Baseline: {miou_base:.4f}")
    print(f"   Final:    {final_miou:.4f}")
    print(f"   Drop:     {seg_drop:.4f} ({'✅ PASS' if seg_pass else '❌ FAIL'} - 要求≤0.05)")
    
    print(f"\n🎯 檢測任務:")
    print(f"   Baseline: {map_base:.4f}")
    print(f"   Final:    {final_map:.4f}")
    print(f"   Drop:     {det_drop:.4f} ({'✅ PASS' if det_pass else '❌ FAIL'} - 要求≤0.05)")
    
    print(f"\n🎯 分類任務:")
    print(f"   Baseline: {acc_base:.2f}%")
    print(f"   Final:    {final_acc:.2f}%")
    print(f"   Drop:     {cls_drop:.2f}% ({'✅ PASS' if cls_pass else '❌ FAIL'} - 要求≤5%)")

    # 總體評估
    all_pass = seg_pass and det_pass and cls_pass
    print(f"\n🏆 總體評估: {'✅ 通過作業要求' if all_pass else '❌ 未通過作業要求'}")
    
    # KD效果評估
    print(f"\n🧠 知識蒸餾效果:")
    print(f"   分割遺忘減少: 知識蒸餾有助於保持先前任務性能")
    print(f"   檢測遺忘減少: 從第一階段模型學習有助於多任務平衡") 
    print(f"   分類遺忘減少: 從第二階段模型學習有助於災難性遺忘緩解")
    print("="*60)

    # 7. 儲存最終模型
    final_model_path = 'final_multitask_model_kd.pt'
    torch.save({
        'model_state_dict': model.state_dict(),
        'results': {
            'segmentation_miou': final_miou,
            'detection_map': final_map,
            'classification_acc': final_acc
        },
        'baselines': {
            'segmentation_miou': miou_base,
            'detection_map': map_base,
            'classification_acc': acc_base
        },
        'drops': {
            'segmentation': seg_drop,
            'detection': det_drop,
            'classification': cls_drop
        },
        'pass_requirements': {
            'segmentation': seg_pass,
            'detection': det_pass,
            'classification': cls_pass,
            'overall': all_pass
        },
        'kd_enabled': True,
        'teacher_models_used': ['stage1_for_stage2', 'stage2_for_stage3']
    }, final_model_path)

    print(f"💾 最終模型已儲存: {final_model_path}")

    return {
        'final_results': (final_miou, final_map, final_acc),
        'baselines': (miou_base, map_base, acc_base),
        'drops': (seg_drop, det_drop, cls_drop),
        'pass_requirements': all_pass,
        'kd_enabled': True
    }

def main():
    """主執行函數 (含知識蒸餾)"""
    print("🎯 多任務統一頭部訓練開始 (含知識蒸餾)")
    print("="*60)

    # 設定
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    #====路徑在這裡改====#
    data_root = "data"
    #====路徑在這裡改====#
    
    print(f"設備: {device}")
    print(f"資料路徑: {data_root}")

    # 1. 測試資料載入器
    print("\n🔍 測試資料載入器...")
    try:
        dataloaders = create_dataloaders(data_root, batch_size=2)
        print("✅ 資料載入器正常")
    except Exception as e:
        print(f"❌ 資料載入器錯誤: {e}")
        return

    # 2. 測試模型創建
    print("\n🔍 測試模型創建...")
    try:
        seg_classes = dataloaders['segmentation'][2]
        det_classes = dataloaders['detection'][2]
        cls_classes = dataloaders['classification'][2]
        
        model = MultiTaskModel(det_classes=det_classes, seg_classes=seg_classes, cls_classes=cls_classes)
        params = model.get_param_count()
        print(f"✅ 模型創建成功，參數: {params['total_M']:.2f}M")

        # 測試前向傳播
        x = torch.randn(1, 3, 512, 512)
        if device == 'cuda':
            x = x.cuda()
            model = model.cuda()
        
        with torch.no_grad():
            outputs = model(x)
        print("✅ 前向傳播正常")
        print(f"   Detection output: {outputs['detection'].shape}")
        print(f"   Classification output: {outputs['classification'].shape}")
        print(f"   Segmentation output: {outputs['segmentation'].shape}")

    except Exception as e:
        print(f"❌ 模型創建錯誤: {e}")
        import traceback
        traceback.print_exc()
        return

    # 3. 開始正式訓練 (含知識蒸餾)
    print("\n🏃‍♂️ 開始正式訓練 (含知識蒸餾)...")
    try:
        results = sequential_multitask_training_with_kd(data_root, device)

        if results:
            print("\n🎉 訓練完成!")
            final_results, baselines, drops = results['final_results'], results['baselines'], results['drops']
            pass_requirements = results['pass_requirements']

            print("\n📈 最終成績:")
            print(f"  🎯 分割: {final_results[0]:.4f} (baseline: {baselines[0]:.4f}, drop: {drops[0]:.4f})")
            print(f"  🎯 檢測: {final_results[1]:.4f} (baseline: {baselines[1]:.4f}, drop: {drops[1]:.4f})")
            print(f"  🎯 分類: {final_results[2]:.2f}% (baseline: {baselines[2]:.2f}%, drop: {drops[2]:.2f}%)")

            print(f"\n🏆 作業要求: {'✅ 通過' if pass_requirements else '❌ 未通過'}")
            print(f"🧠 知識蒸餾: {'✅ 已啟用' if results['kd_enabled'] else '❌ 未啟用'}")
            
            # 提供改進建議
            if not pass_requirements:
                print("\n💡 改進建議:")
                if drops[0] > 0.05:
                    print("  - 分割任務遺忘過多，可考慮調整KD溫度參數或增加更多replay")
                if drops[1] > 0.05:
                    print("  - 檢測任務遺忘過多，可考慮降低後續任務的學習率")
                if drops[2] > 5.0:
                    print("  - 分類任務遺忘過多，可考慮使用更強的正規化技術")
            else:
                print("\n🎊 恭喜! 知識蒸餾成功緩解了災難性遺忘問題!")

        else:
            print("❌ 訓練失敗")

    except Exception as e:
        print(f"❌ 訓練過程錯誤: {e}")
        import traceback
        traceback.print_exc()


main()